# Notebook 4: Outbox & CDC Gotchas You Only Learn In Production

The previous notebooks covered the *happy path*. This one walks through issues teams actually run into and simple mitigations.

## Setup

```bash
cd 04-patterns/outbox-and-cdc
docker compose up -d
uv sync
```

In [1]:
import psycopg, json, uuid
DSN = 'host=localhost port=5432 user=demo password=demo dbname=outbox_demo'

with psycopg.connect(DSN, autocommit=True) as conn:
    conn.execute('DROP TABLE IF EXISTS outbox')
    conn.execute('''
        CREATE TABLE outbox (
            id             BIGSERIAL PRIMARY KEY,
            event_id       UUID NOT NULL UNIQUE,
            aggregate_id   TEXT NOT NULL,
            topic          TEXT NOT NULL,
            schema_version INTEGER NOT NULL DEFAULT 1,
            payload        JSONB NOT NULL,
            created_at     TIMESTAMPTZ NOT NULL DEFAULT now(),
            published_at   TIMESTAMPTZ,
            attempts       INTEGER NOT NULL DEFAULT 0,
            last_error     TEXT
        )''')
    conn.execute('CREATE INDEX outbox_unpublished_idx ON outbox(id) WHERE published_at IS NULL')
print('outbox ready')

outbox ready


## 1. Ordering: global vs per-entity

`ORDER BY id` gives global insertion order *from one publisher*. Run **multiple** publisher workers and events for the same order can arrive at the bus out of order, because two workers can interleave.

The fix is the same trick Kafka uses everywhere: **partition by the entity**. The outbox carries `aggregate_id` (e.g. `order:42`) as the Kafka message key - all events for order 42 go to the same partition and stay ordered *relative to each other*. Different orders can be processed in parallel.

In [2]:
def emit(order_id, topic, payload):
    with psycopg.connect(DSN) as conn:
        with conn.transaction():
            conn.execute(
                '''INSERT INTO outbox(event_id, aggregate_id, topic, payload)
                   VALUES (%s, %s, %s, %s::jsonb)''',
                (str(uuid.uuid4()), f'order:{order_id}', topic, json.dumps(payload)),
            )

# Two orders interleaved
emit(1, 'order.placed',  {'order_id': 1})
emit(2, 'order.placed',  {'order_id': 2})
emit(1, 'order.paid',    {'order_id': 1})
emit(2, 'order.paid',    {'order_id': 2})
emit(1, 'order.shipped', {'order_id': 1})

with psycopg.connect(DSN) as conn:
    rows = conn.execute(
        'SELECT aggregate_id, topic FROM outbox ORDER BY id'
    ).fetchall()
for r in rows: print(r)
print()
print('>> Send to Kafka with key=aggregate_id -> per-order events stay ordered')

('order:1', 'order.placed')
('order:2', 'order.placed')
('order:1', 'order.paid')
('order:2', 'order.paid')
('order:1', 'order.shipped')

>> Send to Kafka with key=aggregate_id -> per-order events stay ordered


## 2. Poison messages: one bad row blocks everyone

A naive publisher does `ORDER BY id ... publish or crash`. If event #7 has a payload the consumer rejects (bad schema, missing field, bug), the publisher either:

- loops forever on event #7, or
- crashes and restarts on event #7 - head-of-line blocking.

Mitigations:

- Track **`attempts`** on each row; after N failures, route it to a dead-letter column or table and move on.
- Record **`last_error`** so humans can investigate.

In [3]:
bus = []
DEAD = []  # dead-letter
MAX_ATTEMPTS = 3

def flaky_publish(event_id, payload):
    # Simulate a poison row: order 2 events fail
    if 'order_id' in payload and payload['order_id'] == 2:
        raise ValueError('bad schema for order 2')
    bus.append((event_id, payload))

def publish_round(batch_size=10):
    sent = 0
    dead = 0
    with psycopg.connect(DSN) as conn:
        with conn.transaction():
            rows = conn.execute('''
                SELECT id, event_id, payload, attempts FROM outbox
                WHERE published_at IS NULL
                ORDER BY id
                LIMIT %s FOR UPDATE SKIP LOCKED
            ''', (batch_size,)).fetchall()
            for row_id, event_id, payload, attempts in rows:
                try:
                    flaky_publish(str(event_id), payload)
                    conn.execute('UPDATE outbox SET published_at = now() WHERE id = %s', (row_id,))
                    sent += 1
                except Exception as e:
                    new_attempts = attempts + 1
                    if new_attempts >= MAX_ATTEMPTS:
                        DEAD.append((str(event_id), payload, str(e)))
                        conn.execute(
                            'UPDATE outbox SET published_at = now(), attempts = %s, last_error = %s WHERE id = %s',
                            (new_attempts, f'DEAD: {e}', row_id),
                        )
                        dead += 1
                    else:
                        conn.execute(
                            'UPDATE outbox SET attempts = %s, last_error = %s WHERE id = %s',
                            (new_attempts, str(e), row_id),
                        )
    return sent, dead

# Run several rounds so the poison row retries and eventually goes to DEAD
for _ in range(5):
    s, d = publish_round()
    print(f'round: sent={s} dead={d}')

print()
print('live bus :')
for e in bus: print(' ', e)
print('dead-letter:')
for e in DEAD: print(' ', e)

round: sent=3 dead=0
round: sent=0 dead=0
round: sent=0 dead=2
round: sent=0 dead=0
round: sent=0 dead=0

live bus :
  ('6bd88188-58eb-4e9f-aaba-eedaff2ba0db', {'order_id': 1})
  ('7a27a0fd-dabd-4397-8e1d-cbf1b21d4f49', {'order_id': 1})
  ('629db38f-4779-41a0-9d86-35b1ef6e2cb5', {'order_id': 1})
dead-letter:
  ('73653d75-0937-4db8-aff6-9d741e08f1f3', {'order_id': 2}, 'bad schema for order 2')
  ('b073f9a7-a086-468d-9b05-e722cb67bdef', {'order_id': 2}, 'bad schema for order 2')


## 3. Schema evolution

Events are a **public contract**. Once `order.placed v1` is in production, consumers depend on its shape. Breaking that silently is how fleets of services catch fire at 3am.

Defensive habits:

- Include **`schema_version`** on every event (we added it to the table above).
- **Only add optional fields** within a version. For breaking changes, bump to v2 and keep publishing v1 until consumers migrate.
- Consider a schema registry (Avro / Protobuf / JSON-Schema) for anything real.

In [4]:
with psycopg.connect(DSN) as conn:
    with conn.transaction():
        # v1 event - today
        conn.execute(
            '''INSERT INTO outbox(event_id, aggregate_id, topic, schema_version, payload)
               VALUES (%s, %s, %s, %s, %s::jsonb)''',
            (str(uuid.uuid4()), 'order:42', 'order.placed', 1,
             json.dumps({'order_id': 42, 'total': 25})),
        )
    with conn.transaction():
        # v2 event - later, adds currency
        conn.execute(
            '''INSERT INTO outbox(event_id, aggregate_id, topic, schema_version, payload)
               VALUES (%s, %s, %s, %s, %s::jsonb)''',
            (str(uuid.uuid4()), 'order:43', 'order.placed', 2,
             json.dumps({'order_id': 43, 'total': 25, 'currency': 'USD'})),
        )

with psycopg.connect(DSN) as conn:
    for r in conn.execute(
        'SELECT topic, schema_version, payload FROM outbox WHERE topic=%s ORDER BY id',
        ('order.placed',),
    ).fetchall():
        print(r)

('order.placed', 1, {'order_id': 1})
('order.placed', 1, {'order_id': 2})
('order.placed', 1, {'total': 25, 'order_id': 42})
('order.placed', 2, {'total': 25, 'currency': 'USD', 'order_id': 43})


## 4. Exactly-once is a lie - design for at-least-once

With any of these patterns (outbox polling *or* CDC), the publisher can crash after shipping an event and before marking it published, so consumers will occasionally see duplicates.

Treat this as a given. Make consumers **idempotent** on `event_id`:

- Keep a `processed_events(event_id PRIMARY KEY)` table and insert into it inside the same transaction as the side-effect. The second delivery fails on the unique constraint and is safely ignored.
- Or make the side-effect naturally idempotent (`UPSERT`, set-based state, etc.).

## 5. Where to go next

- **Debezium** - production-grade CDC for Postgres, MySQL, MongoDB, and more. Plugs into Kafka Connect.
- **Debezium Outbox Event Router** - reads your outbox table via CDC, strips off the row-level change shape, and publishes clean domain events keyed by `aggregate_id`. The best of both worlds.
- **Kafka transactions / idempotent producer** - reduce duplicates *between producer and Kafka*, but you still need consumer-side dedup for end-to-end safety.
- **Watermarks and exactly-once sinks** (Flink, Kafka Streams) - for analytics pipelines that need strict semantics.